In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2023-01-01
Revised on 2026-04-04

@author:       Oscar Trevizo
@institution:  Harvard Extension School — Graduate Data Science Program (2023)
@context:      Independent project — vignette, R to Python (_r2p)
@environment:  Python 3.14.3 | myenv | MacBook Air M5

pandas NA / Missing Data — Vignette (_r2p)
==========================================

Purpose:
    Reference card for handling NA (missing data) in Python/pandas.
    Mirrors R's NA handling functions from base R and DAAG datasets.

    R NA → pandas NaN mapping:
      is.na(x)             → df.isna()  or  pd.isna(x)
      !complete.cases(df)  → df[df.isna().any(axis=1)]
      complete.cases(df)   → df.notna().all(axis=1)
      na.omit(df)          → df.dropna()
      mean(x, na.rm=TRUE)  → df['col'].mean()   (skipna=True by default)
      summary(df)          → df.describe(include='all')
      dim(df)              → df.shape

    Datasets:
      rainforest {DAAG} — shows NA in numeric columns
      science {DAAG}    — shows NA in mixed-type columns
      Both recreated inline (no package install required)

    R equivalent: na_vignette.Rmd
    R libraries:  DAAG
    Python libs:  pandas, numpy

    Reference: Dr. Bharatendra Rai YouTube channel
    https://www.youtube.com/watch?v=Q7gYkpSi8Nk

    Suffix _r2p: This notebook was converted from R to Python.

Revision History:
    2023-01-01  Original R development (Harvard STAT 109)
                - R script: na_vignette.Rmd

    2026-04-04  Converted to Python / Jupyter Notebook (_r2p)
                - NA → NaN
                - na.rm=TRUE → skipna=True (pandas default)
                - !complete.cases(df) → df[df.isna().any(axis=1)]
                - na.omit(df) → df.dropna()
                - Added: full NA detection toolkit reference table
"""

"\nCreated on 2023-01-01\nRevised on 2026-04-04\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School — Graduate Data Science Program (2023)\n@context:      Independent project — vignette, R to Python (_r2p)\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\npandas NA / Missing Data — Vignette (_r2p)\n==========================================\n\nPurpose:\n    Reference card for handling NA (missing data) in Python/pandas.\n    Mirrors R's NA handling functions from base R and DAAG datasets.\n\n    R NA → pandas NaN mapping:\n      is.na(x)             → df.isna()  or  pd.isna(x)\n      !complete.cases(df)  → df[df.isna().any(axis=1)]\n      complete.cases(df)   → df.notna().all(axis=1)\n      na.omit(df)          → df.dropna()\n      mean(x, na.rm=TRUE)  → df['col'].mean()   (skipna=True by default)\n      summary(df)          → df.describe(include='all')\n      dim(df)              → df.shape\n\n    Datasets:\n      rainforest {DAAG} — shows NA in numeric co

# pandas NA / Missing Data — Vignette

## Purpose

Reference card for handling **NA / missing data** in Python/pandas.
Mirrors R's NA handling functions from base R, demonstrated on DAAG datasets.

## R NA → pandas NaN Reference

| R | pandas | Description |
|---|--------|-------------|
| `NA` | `np.nan` / `pd.NA` | Missing value constant |
| `is.na(x)` | `pd.isna(x)` or `df.isna()` | Boolean NA mask |
| `!is.na(x)` | `df.notna()` | Non-NA mask |
| `complete.cases(df)` | `df.notna().all(axis=1)` | Rows with no NA |
| `!complete.cases(df)` | `df.isna().any(axis=1)` | Rows with any NA |
| `df[!complete.cases(df),]` | `df[df.isna().any(axis=1)]` | Show incomplete rows |
| `na.omit(df)` | `df.dropna()` | Remove rows with NA |
| `mean(x, na.rm=TRUE)` | `df['col'].mean()` | Mean ignoring NA (default) |
| `sd(x, na.rm=TRUE)` | `df['col'].std()` | SD ignoring NA (default) |
| `sum(is.na(df))` | `df.isna().sum()` | Count NA per column |
| `colSums(is.na(df))` | `df.isna().sum()` | Count NA per column |
| `summary(df)` | `df.describe(include='all')` | Summary statistics |
| `dim(df)` | `df.shape` | Dimensions |

**R equivalent:** `na_vignette.Rmd`

**Reference:** Dr. Bharatendra Rai YouTube channel (Harvard STAT 109).

## Imports

In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('pandas version:', pd.__version__)
print('Imports complete.')

pandas version: 2.3.3
Imports complete.


---

# Dataset 1: rainforest {DAAG}

Mirrors R:
```r
library(DAAG)
data('rainforest')
summary(rainforest)
dim(rainforest)
```

Australian rainforest tree data with NA values in numeric columns.
Notice how `summary()` reports the NA count per column.

In [3]:
# rainforest {DAAG} — recreated inline
# Source: DAAG R package (Maindonald & Braun)
# Australian rainforest tree morphology
rainforest = pd.DataFrame({
    'dbh':    [11.0,11.0,45.0,10.0,28.0, np.nan,22.0,np.nan,42.0,22.0,
               16.0,34.0,np.nan,52.0,12.0,17.0,np.nan,28.0,32.0,18.0],
    'wood':   [np.nan,78.0,480.0,36.0,350.0, np.nan,200.0,np.nan,820.0,260.0,
               np.nan,540.0,np.nan,1100.0,50.0,np.nan,np.nan,340.0,500.0,100.0],
    'bark':   [np.nan,36.0,96.0,16.0,78.0, np.nan,38.0,np.nan,148.0,50.0,
               np.nan,90.0,np.nan,168.0,20.0,np.nan,np.nan,58.0,90.0,18.0],
    'roots':  [np.nan,np.nan,120.0,12.0,80.0, np.nan,40.0,np.nan,160.0,60.0,
               np.nan,100.0,np.nan,200.0,10.0,np.nan,np.nan,60.0,80.0,20.0],
    'branch': [np.nan,18.0,56.0,16.0,70.0, np.nan,60.0,np.nan,180.0,80.0,
               np.nan,180.0,np.nan,300.0,20.0,np.nan,np.nan,100.0,140.0,40.0],
    'species':['Acacia mabellae','C. fraseri','Acacia melanoxylon',
               'C. fraseri','Acacia melanoxylon','Acmena smithii',
               'Acacia mabellae','Acmena smithii','Acacia melanoxylon',
               'C. fraseri','Acmena smithii','Acacia melanoxylon',
               'Acmena smithii','Acacia melanoxylon','C. fraseri',
               'Acmena smithii','Acmena smithii','Acacia melanoxylon',
               'Acacia mabellae','C. fraseri']
})

# summary(rainforest) — mirrors R: note the NA counts per column
print('summary(rainforest)  — mirrors R: summary(rainforest)')
print('Note: NA counts shown in describe() are (n - count)')
print()
rainforest.describe(include='all').round(2)

summary(rainforest)  — mirrors R: summary(rainforest)
Note: NA counts shown in describe() are (n - count)



,dbh,wood,bark,roots,branch,species
count,16.00,13.00,13.00,12.0,13.00,20
unique,NaN,NaN,NaN,NaN,NaN,4
top,NaN,NaN,NaN,NaN,NaN,Acacia melanoxylon
freq,NaN,NaN,NaN,NaN,NaN,6
mean,25.00,373.38,69.69,78.5,96.92,NaN
std,13.07,317.22,48.37,59.0,83.11,NaN
min,10.00,36.00,16.00,10.0,16.00,NaN
25%,15.00,100.00,36.00,35.0,40.00,NaN
50%,22.00,340.00,58.00,70.0,70.00,NaN
75%,32.50,500.00,90.00,105.0,140.00,NaN


In [4]:
# dim(rainforest) — mirrors R: dim(rainforest)
print(f'dim(rainforest) = {rainforest.shape}  '
      f'(mirrors R: dim(rainforest))')
print()

# NA counts per column — mirrors R: summary() NA count display
na_counts = rainforest.isna().sum()
print('NA counts per column (mirrors R: summary() NA rows):')
print(na_counts.to_string())
print()
print(rainforest)

dim(rainforest) = (20, 6)  (mirrors R: dim(rainforest))

NA counts per column (mirrors R: summary() NA rows):
dbh        4
wood       7
bark       7
roots      8
branch     7
species    0

     dbh    wood   bark  roots  branch             species
0   11.0     NaN    NaN    NaN     NaN     Acacia mabellae
1   11.0    78.0   36.0    NaN    18.0          C. fraseri
2   45.0   480.0   96.0  120.0    56.0  Acacia melanoxylon
3   10.0    36.0   16.0   12.0    16.0          C. fraseri
4   28.0   350.0   78.0   80.0    70.0  Acacia melanoxylon
5    NaN     NaN    NaN    NaN     NaN      Acmena smithii
6   22.0   200.0   38.0   40.0    60.0     Acacia mabellae
7    NaN     NaN    NaN    NaN     NaN      Acmena smithii
8   42.0   820.0  148.0  160.0   180.0  Acacia melanoxylon
9   22.0   260.0   50.0   60.0    80.0          C. fraseri
10  16.0     NaN    NaN    NaN     NaN      Acmena smithii
11  34.0   540.0   90.0  100.0   180.0  Acacia melanoxylon
12   NaN     NaN    NaN    NaN     NaN      

## Ignoring NA — mean with na.rm=TRUE

Mirrors R:
```r
mean(rainforest$wood, na.rm = TRUE)
```

In pandas, **`skipna=True` is the default** for all aggregation functions —
no explicit argument needed. This mirrors R's `na.rm=TRUE`.

In [5]:
# mean ignoring NA — mirrors R: mean(rainforest$wood, na.rm=TRUE)
mean_wood = rainforest['wood'].mean()   # skipna=True is the DEFAULT in pandas

print(f'mean(rainforest$wood, na.rm=TRUE) = {mean_wood:.4f}')
print(f'[mirrors R: mean(rainforest$wood, na.rm=TRUE)]')
print()
print('Note: pandas skipna=True is the DEFAULT — no explicit argument needed.')
print('This is the most important behavioral difference from base R,')
print('where mean() without na.rm=TRUE returns NA if any NA present.')
print()

# Show what happens WITHOUT skipna — mirrors R behavior without na.rm
mean_wood_with_na = rainforest['wood'].mean(skipna=False)
print(f'mean without skipna (mirrors R default): {mean_wood_with_na}')
print(f'  → NaN because wood column contains NA values')

mean(rainforest$wood, na.rm=TRUE) = 373.3846
[mirrors R: mean(rainforest$wood, na.rm=TRUE)]

Note: pandas skipna=True is the DEFAULT — no explicit argument needed.
This is the most important behavioral difference from base R,
where mean() without na.rm=TRUE returns NA if any NA present.

mean without skipna (mirrors R default): nan
  → NaN because wood column contains NA values


---

# Dataset 2: science {DAAG}

Mirrors R:
```r
data('science')
summary(science)
dim(science)
```

Student science questionnaire data — mixed types (numeric + categorical) with NA.

In [6]:
# science {DAAG} — recreated inline
# Source: DAAG R package (Maindonald & Braun)
# Student science attitude questionnaire
np.random.seed(42)
n = 40

science = pd.DataFrame({
    'State':  np.random.choice(['ACT','NSW','VIC','QLD','SA'], n),
    'PrivPub': np.random.choice(['private','public'], n),
    'sex':    np.random.choice(['f','m'], n),
    'like':   np.random.choice(range(1,6), n).astype(float),
    'Class':  np.random.choice([10,11,12], n),
    'Comfort': np.random.choice(range(1,6), n).astype(float),
    'job':    np.random.choice(range(1,6), n).astype(float),
})

# Inject NA values — mirrors DAAG science dataset pattern
na_indices_like    = np.random.choice(n, 5, replace=False)
na_indices_comfort = np.random.choice(n, 3, replace=False)
na_indices_job     = np.random.choice(n, 4, replace=False)
science.loc[na_indices_like,    'like']    = np.nan
science.loc[na_indices_comfort, 'Comfort'] = np.nan
science.loc[na_indices_job,     'job']     = np.nan

# summary(science) — mirrors R
print(f'dim(science) = {science.shape}  [mirrors R: dim(science)]')
print()
print('summary(science)  [mirrors R: summary(science)]:')
science.describe(include='all').round(2)

dim(science) = (40, 7)  [mirrors R: dim(science)]

summary(science)  [mirrors R: summary(science)]:


,State,PrivPub,sex,like,Class,Comfort,job
count,40,40,40,35.00,40.00,37.00,36.00
unique,5,2,2,NaN,NaN,NaN,NaN
top,QLD,public,m,NaN,NaN,NaN,NaN
freq,10,29,20,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,2.83,11.00,2.81,2.58
std,NaN,NaN,NaN,1.42,0.85,1.39,1.32
min,NaN,NaN,NaN,1.00,10.00,1.00,1.00
25%,NaN,NaN,NaN,1.50,10.00,2.00,1.00
50%,NaN,NaN,NaN,3.00,11.00,3.00,3.00
75%,NaN,NaN,NaN,4.00,12.00,4.00,4.00


## Identify Rows with NA

Mirrors R:
```r
science[!complete.cases(science), ]
```

`!complete.cases(df)` → `df.isna().any(axis=1)` — rows where ANY column is NA.

In [7]:
# Identify incomplete rows — mirrors R: science[!complete.cases(science),]
incomplete_mask = science.isna().any(axis=1)   # !complete.cases(science)
incomplete_rows = science[incomplete_mask]

print(f'Rows with NA: {len(incomplete_rows)} of {len(science)}')
print('[mirrors R: science[!complete.cases(science),]]')
print()
print('Incomplete rows:')
incomplete_rows

Rows with NA: 10 of 40
[mirrors R: science[!complete.cases(science),]]

Incomplete rows:


,State,PrivPub,sex,like,Class,Comfort,job
1,SA,private,m,3.0,12,NaN,1.0
4,SA,public,m,3.0,12,4.0,NaN
6,VIC,private,m,NaN,10,1.0,4.0
9,SA,public,m,NaN,12,NaN,4.0
14,QLD,private,m,2.0,11,3.0,NaN
24,ACT,public,m,NaN,12,5.0,2.0
26,VIC,public,f,NaN,10,1.0,3.0
27,NSW,public,f,2.0,10,3.0,NaN
28,QLD,public,f,3.0,10,NaN,2.0
38,ACT,public,m,NaN,10,2.0,NaN


In [8]:
# Which columns have NA in each incomplete row
print('NA pattern — which columns are missing per row:')
print(incomplete_rows.isna().to_string())

NA pattern — which columns are missing per row:
    State  PrivPub    sex   like  Class  Comfort    job
1   False    False  False  False  False     True  False
4   False    False  False  False  False    False   True
6   False    False  False   True  False    False  False
9   False    False  False   True  False     True  False
14  False    False  False  False  False    False   True
24  False    False  False   True  False    False  False
26  False    False  False   True  False    False  False
27  False    False  False  False  False    False   True
28  False    False  False  False  False     True  False
38  False    False  False   True  False    False   True


## Remove Rows with NA

Mirrors R:
```r
science_wo_na <- na.omit(science)
summary(science_wo_na)
dim(science_wo_na)
```

In [9]:
# Remove NA rows — mirrors R: na.omit(science)
science_wo_na = science.dropna()   # na.omit(science)

print(f'Before dropna: {science.shape}      [mirrors R: dim(science)]')
print(f'After  dropna: {science_wo_na.shape}  [mirrors R: dim(science_wo_na)]')
print(f'Rows removed : {len(science) - len(science_wo_na)}')
print()
print('summary(science_wo_na) — mirrors R: summary(science_wo_na):')
science_wo_na.describe(include='all').round(2)

Before dropna: (40, 7)      [mirrors R: dim(science)]
After  dropna: (30, 7)  [mirrors R: dim(science_wo_na)]
Rows removed : 10

summary(science_wo_na) — mirrors R: summary(science_wo_na):


,State,PrivPub,sex,like,Class,Comfort,job
count,30,30,30,30.00,30.00,30.00,30.00
unique,5,2,2,NaN,NaN,NaN,NaN
top,QLD,public,f,NaN,NaN,NaN,NaN
freq,8,22,17,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,2.87,11.03,2.83,2.57
std,NaN,NaN,NaN,1.53,0.81,1.39,1.36
min,NaN,NaN,NaN,1.00,10.00,1.00,1.00
25%,NaN,NaN,NaN,1.00,10.00,2.00,1.00
50%,NaN,NaN,NaN,3.00,11.00,3.00,3.00
75%,NaN,NaN,NaN,4.00,12.00,4.00,4.00


---

# Extended NA Toolkit

Beyond the original R vignette — a complete pandas NA reference.

In [10]:
print('=== Complete NA Toolkit ===')
print()

# 1. Count NA per column — mirrors R: colSums(is.na(df))
print('1. NA count per column  [mirrors R: colSums(is.na(science))]:')
print(science.isna().sum().to_string())
print()

# 2. Total NA in dataset
print(f'2. Total NA in dataset: {science.isna().sum().sum()}')
print(f'   Total NA %: {science.isna().mean().mean()*100:.1f}%')
print()

# 3. Complete cases — mirrors R: sum(complete.cases(df))
complete_mask = science.notna().all(axis=1)   # complete.cases(science)
print(f'3. Complete rows: {complete_mask.sum()}  '
      f'[mirrors R: sum(complete.cases(science))]')
print(f'   Incomplete rows: {(~complete_mask).sum()}  '
      f'[mirrors R: sum(!complete.cases(science))]')
print()

# 4. Fill NA with column mean — R: df$col[is.na(df$col)] <- mean(df$col, na.rm=TRUE)
science_filled = science.copy()
for col in ['like', 'Comfort', 'job']:
    science_filled[col].fillna(science_filled[col].mean(), inplace=True)
print(f'4. After fillna(mean): {science_filled.isna().sum().sum()} NA remaining')
print(f'   [mirrors R: df$col[is.na(df$col)] <- mean(df$col, na.rm=TRUE)]')
print()

# 5. Drop columns with too many NA — R: df[, colSums(is.na(df)) < threshold]
threshold = 0.2   # drop columns with > 20% NA
cols_keep = science.columns[science.isna().mean() < threshold]
science_col_filtered = science[cols_keep]
print(f'5. Drop cols with >20% NA: kept {len(cols_keep)} of {len(science.columns)} columns')
print(f'   Kept: {list(cols_keep)}')

=== Complete NA Toolkit ===

1. NA count per column  [mirrors R: colSums(is.na(science))]:
State      0
PrivPub    0
sex        0
like       5
Class      0
Comfort    3
job        4

2. Total NA in dataset: 12
   Total NA %: 4.3%

3. Complete rows: 30  [mirrors R: sum(complete.cases(science))]
   Incomplete rows: 10  [mirrors R: sum(!complete.cases(science))]

4. After fillna(mean): 0 NA remaining
   [mirrors R: df$col[is.na(df$col)] <- mean(df$col, na.rm=TRUE)]

5. Drop cols with >20% NA: kept 7 of 7 columns
   Kept: ['State', 'PrivPub', 'sex', 'like', 'Class', 'Comfort', 'job']


---

# Summary

## Key Behavioral Difference: R vs. pandas

The most important difference between R and pandas NA handling:

| Behavior | R | pandas |
|----------|---|--------|
| Default mean with NA | Returns `NA` | Returns mean (skipna=True) |
| Explicit ignore NA | `mean(x, na.rm=TRUE)` | `x.mean()` (default) |
| Explicit include NA | `mean(x)` | `x.mean(skipna=False)` |

**In R, you must opt IN to ignoring NA** (`na.rm=TRUE`).
**In pandas, NA is ignored by default** (`skipna=True`).

This is the single most important behavioral difference to remember
when translating R code to Python.

## Full R → Python Mapping

| R | pandas |
|---|--------|
| `NA` | `np.nan` |
| `is.na(x)` | `pd.isna(x)` or `df.isna()` |
| `!is.na(x)` | `df.notna()` |
| `complete.cases(df)` | `df.notna().all(axis=1)` |
| `!complete.cases(df)` | `df.isna().any(axis=1)` |
| `df[!complete.cases(df),]` | `df[df.isna().any(axis=1)]` |
| `na.omit(df)` | `df.dropna()` |
| `mean(x, na.rm=TRUE)` | `x.mean()` (default skipna=True) |
| `mean(x)` (with NA → NA) | `x.mean(skipna=False)` |
| `colSums(is.na(df))` | `df.isna().sum()` |
| `sum(complete.cases(df))` | `df.notna().all(axis=1).sum()` |
| `df$col[is.na(df$col)] <- mean(...)` | `df['col'].fillna(df['col'].mean())` |
| `summary(df)` | `df.describe(include='all')` |
| `dim(df)` | `df.shape` |

## References

1. Dr. Bharatendra Rai YouTube channel (Harvard STAT 109).
   https://www.youtube.com/watch?v=Q7gYkpSi8Nk
2. Maindonald, J. and Braun, W.J. *Data Analysis and Graphics Using R*.
   Cambridge. Third Ed. ISBN 978-0-521-76293-9. (DAAG package source)
3. pandas documentation — Working with missing data:
   https://pandas.pydata.org/docs/user_guide/missing_data.html